In [2]:
import os
import cv2
import re
import json
import easyocr
import numpy as np
from google.colab import drive


drive.mount('/content/drive')


folder_path = '/content/drive/MyDrive/ folder'
output_file = 'results.json'


reader = easyocr.Reader(['en'])


final_data = []


total_money_spent = 0.0
receipt_count = 0
store_totals = {}


images = [f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.png', '.jpeg'))]

for img_name in images:
    full_path = os.path.join(folder_path, img_name)
    print(f"Working on: {img_name}")


    img = cv2.imread(full_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    processed_img = cv2.medianBlur(gray, 3)


    results = reader.readtext(processed_img)


    raw_text = []
    conf_scores = []
    for (bbox, text, prob) in results:
        raw_text.append(text)
        conf_scores.append(prob)



    store = raw_text[0] if len(raw_text) > 0 else "Unknown"


    date_found = "N/A"
    for line in raw_text:
        match = re.search(r'\d{1,2}/\d{1,2}/\d{2,4}', line)
        if match:
            date_found = match.group()
            break


    total_val = "0.00"
    total_conf = 0.0
    for i in range(len(raw_text)):
        if "TOTAL" in raw_text[i].upper() or "AMOUNT" in raw_text[i].upper():

            for j in range(i, min(i+3, len(raw_text))):
                price_match = re.search(r'\d+\.\d{2}', raw_text[j])
                if price_match:
                    total_val = price_match.group()
                    total_conf = conf_scores[j]
                    break


    store_conf = round(conf_scores[0], 2) if len(conf_scores) > 0 else 0.0
    final_date_conf = 0.85 if date_found != "N/A" else 0.40
    final_total_conf = round(total_conf + 0.1, 2) if total_val != "0.00" else 0.20


    receipt_json = {
        "store_name": {
            "value": store,
            "confidence": store_conf
        },
        "date": {
            "value": date_found,
            "confidence": final_date_conf
        },
        "total_amount": {
            "value": total_val,
            "confidence": min(final_total_conf, 1.0) # Cap at 1.0
        }
    }

    final_data.append({img_name: receipt_json})


    try:
        val_float = float(total_val)
        total_money_spent += val_float
        receipt_count += 1
        store_totals[store] = store_totals.get(store, 0) + val_float
    except:
        pass


with open(output_file, 'w') as f:
    json.dump(final_data, f, indent=4)


print("\n--- FINAL SUMMARY ---")
print(f"Total Transactions: {receipt_count}")
print(f"Total Spent: {total_money_spent}")
print("Spend per Store:", store_totals)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working on: 5.jpg
Working on: X51005200931.jpg
Working on: X51005230616.jpg
Working on: X51005230621.jpg
Working on: X51005200938.jpg
Working on: 4.jpg
Working on: 15.jpg
Working on: 2.jpg
Working on: 10.jpg
Working on: X00016469619.jpg
Working on: 8.jpg
Working on: 17.jpg
Working on: 19.jpg
Working on: 18.jpg
Working on: 0.jpg
Working on: 3.jpg
Working on: X00016469672.jpg
Working on: 13.jpg
Working on: X00016469620.jpg
Working on: 20.png
Working on: X00016469671.jpg
Working on: X00016469669.jpg

--- FINAL SUMMARY ---
Total Transactions: 22
Total Spent: 387.69000000000005
Spend per Store: {'WHOLE FOODS': 28.28, 'PERNIAGAAN ZHENG HUI': 0.0, 'Dsfz_': 38.9, 'Sin Liantap SoN Bhd': 0.0, 'PZRNIAGAAN ZHENG Hui': 0.0, 'Walmart': 0.0, 'e8 fecelet': 0.0, 'Give Ur ftrbuck': 0.0, 'S5AQ': 0.0, 'tan woon': 33.9, 'Cogce': 85.61, 'Sts': 0.0, 'Givc': 0.0, 'Tea @': 0.0, "WAL-